In [1]:
import torch
from modelscope import AutoModelForCausalLM

d:\Anaconda\envs\torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step1 加载模型

In [2]:
print("正在加载GPT2模型...")
model_dir = '../models/gpt2'
model = AutoModelForCausalLM.from_pretrained(model_dir)
model.eval()
print("GPT-2 模型加载完毕")

正在加载GPT2模型...
GPT-2 模型加载完毕


### Step2 查看所有命名模块

In [3]:
for name, module in model.named_modules():
    if name:  # 跳过根模块
        print(f"{name}: {module.__class__.__name__}")
        # 如果有参数,显示参数形状
        for param_name, param in module.named_parameters(recurse=False):
            print(f"  └─ {param_name}: {param.shape}")

transformer: GPT2Model
transformer.wte: Embedding
  └─ weight: torch.Size([50257, 768])
transformer.wpe: Embedding
  └─ weight: torch.Size([1024, 768])
transformer.drop: Dropout
transformer.h: ModuleList
transformer.h.0: GPT2Block
transformer.h.0.ln_1: LayerNorm
  └─ weight: torch.Size([768])
  └─ bias: torch.Size([768])
transformer.h.0.attn: GPT2SdpaAttention
transformer.h.0.attn.c_attn: Conv1D
  └─ weight: torch.Size([768, 2304])
  └─ bias: torch.Size([2304])
transformer.h.0.attn.c_proj: Conv1D
  └─ weight: torch.Size([768, 768])
  └─ bias: torch.Size([768])
transformer.h.0.attn.attn_dropout: Dropout
transformer.h.0.attn.resid_dropout: Dropout
transformer.h.0.ln_2: LayerNorm
  └─ weight: torch.Size([768])
  └─ bias: torch.Size([768])
transformer.h.0.mlp: GPT2MLP
transformer.h.0.mlp.c_fc: Conv1D
  └─ weight: torch.Size([768, 3072])
  └─ bias: torch.Size([3072])
transformer.h.0.mlp.c_proj: Conv1D
  └─ weight: torch.Size([3072, 768])
  └─ bias: torch.Size([768])
transformer.h.0.mlp.act:

### Step3 直接访问特定层

In [4]:
# 访问词嵌入层
print("\n1️⃣ 词嵌入层 (Token Embedding):")
wte = model.transformer.wte
print(f"   类型: {wte.__class__.__name__}")
print(f"   权重形状: {wte.weight.shape}")  # [vocab_size, hidden_dim]
print(f"   词汇表大小: {wte.weight.shape[0]}")
print(f"   隐藏维度: {wte.weight.shape[1]}")

# # 访问位置嵌入层
# print("\n2️⃣ 位置嵌入层 (Position Embedding):")
# wpe = model.transformer.wpe
# print(f"   类型: {wpe.__class__.__name__}")
# print(f"   权重形状: {wpe.weight.shape}")  # [max_position, hidden_dim]
# print(f"   最大位置: {wpe.weight.shape[0]}")

# # 访问第一个Transformer Block
# print("\n3️⃣ 第一个Transformer Block (h.0):")
# block_0 = model.transformer.h[0]
# print(f"   类型: {block_0.__class__.__name__}")

# # 访问注意力层
# print("\n   📌 注意力层 (attn):")
# attn = block_0.attn
# print(f"      类型: {attn.__class__.__name__}")
# print(f"      c_attn权重: {attn.c_attn.weight.shape}")  # [hidden_dim, 3*hidden_dim]
# print(f"      c_proj权重: {attn.c_proj.weight.shape}")  # [hidden_dim, hidden_dim]

# # 访问MLP层
# print("\n   📌 MLP层:")
# mlp = block_0.mlp
# print(f"      类型: {mlp.__class__.__name__}")
# print(f"      c_fc权重: {mlp.c_fc.weight.shape}")    # [hidden_dim, 4*hidden_dim]
# print(f"      c_proj权重: {mlp.c_proj.weight.shape}")  # [4*hidden_dim, hidden_dim]
# print(f"      激活函数: {mlp.act.__class__.__name__}")

# # 访问LayerNorm
# print("\n   📌 LayerNorm层:")
# print(f"      ln_1权重: {block_0.ln_1.weight.shape}")
# print(f"      ln_2权重: {block_0.ln_2.weight.shape}")

# # 访问输出层
# print("\n4️⃣ 语言模型头 (lm_head):")
# lm_head = model.lm_head
# print(f"   类型: {lm_head.__class__.__name__}")
# print(f"   权重形状: {lm_head.weight.shape}")  # [vocab_size, hidden_dim]


1️⃣ 词嵌入层 (Token Embedding):
   类型: Embedding
   权重形状: torch.Size([50257, 768])
   词汇表大小: 50257
   隐藏维度: 768


### Step4 查看所有参数统计

In [5]:
total_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
    print(f"{name:60s} | Shape: {str(param.shape):25s} | 参数量: {param.numel():>12,}")

print("-" * 80)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
print(f"参数大小: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

transformer.wte.weight                                       | Shape: torch.Size([50257, 768])  | 参数量:   38,597,376
transformer.wpe.weight                                       | Shape: torch.Size([1024, 768])   | 参数量:      786,432
transformer.h.0.ln_1.weight                                  | Shape: torch.Size([768])         | 参数量:          768
transformer.h.0.ln_1.bias                                    | Shape: torch.Size([768])         | 参数量:          768
transformer.h.0.attn.c_attn.weight                           | Shape: torch.Size([768, 2304])   | 参数量:    1,769,472
transformer.h.0.attn.c_attn.bias                             | Shape: torch.Size([2304])        | 参数量:        2,304
transformer.h.0.attn.c_proj.weight                           | Shape: torch.Size([768, 768])    | 参数量:      589,824
transformer.h.0.attn.c_proj.bias                             | Shape: torch.Size([768])         | 参数量:          768
transformer.h.0.ln_2.weight                                  | Shape: to

### Step5 访问特定层的参数值

In [6]:
# 查看第一个token的embedding向量
print("\n示例1: 查看token ID=0的embedding向量(前10个值):")
token_0_embedding = model.transformer.wte.weight[0, :10]
print(f"{token_0_embedding}")

# 查看第一层注意力的部分权重
print("\n示例2: 查看第一层注意力c_attn的部分权重(前5x5):")
attn_weights = model.transformer.h[0].attn.c_attn.weight[:5, :5]
print(f"{attn_weights}")


示例1: 查看token ID=0的embedding向量(前10个值):
tensor([-0.1101, -0.0393,  0.0331,  0.1338, -0.0485, -0.0789, -0.2398, -0.0895,
         0.0253, -0.1074], grad_fn=<SliceBackward0>)

示例2: 查看第一层注意力c_attn的部分权重(前5x5):
tensor([[-0.4738, -0.2614, -0.0978, -0.3499,  0.2243],
        [ 0.0874,  0.1473,  0.2387,  0.2884,  0.6491],
        [ 0.0039,  0.0695,  0.3668, -0.0726, -0.1496],
        [ 0.2215, -0.1884, -0.0141,  0.4252,  0.2499],
        [-0.0947,  0.1678, -0.0143,  0.5632,  0.2514]],
       grad_fn=<SliceBackward0>)


### Step6 使用state_dict查看所有参数键

In [7]:
state_dict = model.state_dict()
print(f"\n总共有 {len(state_dict)} 个参数张量:")
for i, (key, value) in enumerate(state_dict.items(), 1):
    print(f"{i:3d}. {key:60s} | {value.shape}")


总共有 149 个参数张量:
  1. transformer.wte.weight                                       | torch.Size([50257, 768])
  2. transformer.wpe.weight                                       | torch.Size([1024, 768])
  3. transformer.h.0.ln_1.weight                                  | torch.Size([768])
  4. transformer.h.0.ln_1.bias                                    | torch.Size([768])
  5. transformer.h.0.attn.c_attn.weight                           | torch.Size([768, 2304])
  6. transformer.h.0.attn.c_attn.bias                             | torch.Size([2304])
  7. transformer.h.0.attn.c_proj.weight                           | torch.Size([768, 768])
  8. transformer.h.0.attn.c_proj.bias                             | torch.Size([768])
  9. transformer.h.0.ln_2.weight                                  | torch.Size([768])
 10. transformer.h.0.ln_2.bias                                    | torch.Size([768])
 11. transformer.h.0.mlp.c_fc.weight                              | torch.Size([768, 3072])
 12. tr

### Step7 按层分组查看

In [8]:
num_layers = len(model.transformer.h)
print(f"\nGPT-2共有 {num_layers} 个Transformer层\n")

for i in range(num_layers):
    print(f"第 {i} 层:")
    block = model.transformer.h[i]
    
    # 统计该层参数量
    layer_params = sum(p.numel() for p in block.parameters())
    print(f"  参数量: {layer_params:,}")
    
    # 列出该层所有参数
    for name, param in block.named_parameters():
        print(f"    {name:40s} | {param.shape}")
    print()


GPT-2共有 12 个Transformer层

第 0 层:
  参数量: 7,087,872
    ln_1.weight                              | torch.Size([768])
    ln_1.bias                                | torch.Size([768])
    attn.c_attn.weight                       | torch.Size([768, 2304])
    attn.c_attn.bias                         | torch.Size([2304])
    attn.c_proj.weight                       | torch.Size([768, 768])
    attn.c_proj.bias                         | torch.Size([768])
    ln_2.weight                              | torch.Size([768])
    ln_2.bias                                | torch.Size([768])
    mlp.c_fc.weight                          | torch.Size([768, 3072])
    mlp.c_fc.bias                            | torch.Size([3072])
    mlp.c_proj.weight                        | torch.Size([3072, 768])
    mlp.c_proj.bias                          | torch.Size([768])

第 1 层:
  参数量: 7,087,872
    ln_1.weight                              | torch.Size([768])
    ln_1.bias                                | torch.S